# Phase 1 EDA — Supplementary Investigation

This notebook addresses all gaps identified in the **Phase 0.5 and Phase 1 review**
(`documentation/supplementary_docs/0_1_review.md`).

**Sections:**
1. FIRST_BREAK_TIME vs SPARE1 comparison
2. MODELLED_BREAK_TIME investigation
3. FIRST_BREAK_AMPLIT and FIRST_BREAK_VELOCITY summary
4. Spatial coordinate sanity check
5. Resampling comparison for Lalor
6. SNR proxy redefinition and visualization example reclassification
7. Brunswick gather width memory estimate

**Outputs:**
- `results/eda/supplementary_report.json`
- `results/eda_plots/supplementary/*.png`
- Updated `results/visualization_examples.json`


In [3]:
# ============================================================
# Cell 1 — Configuration & Setup
# ============================================================
import sys, os, json, time, warnings
import numpy as np
import h5py
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for Colab
import matplotlib.pyplot as plt
import yaml

# Mount Drive (idempotent)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Load project config
REPO_ROOT = "/content/drive/MyDrive/seismic-first-break-picking" if IN_COLAB else os.getcwd()
config_path = os.path.join(REPO_ROOT, "configs", "datasets.yaml")
with open(config_path) as f:
    CFG = yaml.safe_load(f)

ASSET_NAMES = CFG["assets"]["names"]
HDF5_GROUP = "TRACE_DATA/DEFAULT"

# Create supplementary plots directory
SUPP_PLOT_DIR = os.path.join(REPO_ROOT, CFG["paths"]["eda_plots"], "supplementary")
SUPP_JSON_DIR = os.path.join(REPO_ROOT, CFG["paths"]["eda_json"])
os.makedirs(SUPP_PLOT_DIR, exist_ok=True)
os.makedirs(SUPP_JSON_DIR, exist_ok=True)

# Import shared utilities
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from data.eda_utils import open_hdf5, apply_segy_scale, make_json_safe

# Helper: open all assets
def open_all_assets():
    handles = {}
    for name in ASSET_NAMES:
        meta = CFG["asset_meta"][name]
        path = os.path.join(REPO_ROOT, CFG["paths"]["extracted"], meta["extracted_file"])
        f, grp = open_hdf5(path)
        handles[name] = {"file": f, "grp": grp, "meta": meta}
    return handles

# Supplementary report accumulator
supp_report = {}

print("Configuration loaded. Assets:", ASSET_NAMES)
print("Supplementary plots dir:", SUPP_PLOT_DIR)


Mounted at /content/drive
Configuration loaded. Assets: ['brunswick', 'halfmile', 'lalor', 'sudbury']
Supplementary plots dir: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/supplementary


## Section 1 — FIRST_BREAK_TIME vs SPARE1 Comparison

For each asset, load both FIRST_BREAK_TIME and SPARE1 for a random sample of 5000 labeled traces.
Compute the difference. Plot a histogram of the differences.
If they are identical, document that SPARE1 is confirmed as the sole ground truth field.
If they differ, document how and by how much.

**Key context:** Phase 0.5 found `n_both_labeled: 0` for all assets, meaning that for every asset,
the existing comparison function found zero traces where both SPARE1 > 0 AND FIRST_BREAK_TIME > 0.
This section investigates WHY that happened and whether FIRST_BREAK_TIME contains anything useful.


In [4]:
# ============================================================
# Cell 2 — Section 1: FIRST_BREAK_TIME vs SPARE1
# ============================================================
handles = open_all_assets()
section1_results = {}

N_SAMPLE = 5000
rng = np.random.RandomState(42)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, name in enumerate(ASSET_NAMES):
    grp = handles[name]["grp"]
    meta = handles[name]["meta"]

    spare1 = grp["SPARE1"][:].ravel().astype(np.float64)

    result = {"asset": name}

    if "FIRST_BREAK_TIME" in grp:
        fbt = grp["FIRST_BREAK_TIME"][:].ravel().astype(np.float64)
        result["fbt_present"] = True
        result["fbt_shape"] = list(grp["FIRST_BREAK_TIME"].shape)
        result["fbt_dtype"] = str(grp["FIRST_BREAK_TIME"].dtype)
        result["spare1_shape"] = list(grp["SPARE1"].shape)
        result["spare1_dtype"] = str(grp["SPARE1"].dtype)

        # Characterize FIRST_BREAK_TIME values
        fbt_zero = int(np.sum(fbt == 0))
        fbt_neg = int(np.sum(fbt < 0))
        fbt_pos = int(np.sum(fbt > 0))
        result["fbt_count_zero"] = fbt_zero
        result["fbt_count_negative"] = fbt_neg
        result["fbt_count_positive"] = fbt_pos
        result["fbt_total"] = len(fbt)

        # Characterize SPARE1 values
        s1_zero = int(np.sum(spare1 == 0))
        s1_neg = int(np.sum(spare1 < 0))
        s1_pos = int(np.sum(spare1 > 0))
        result["spare1_count_zero"] = s1_zero
        result["spare1_count_negative"] = s1_neg
        result["spare1_count_positive"] = s1_pos
        result["spare1_total"] = len(spare1)

        # Check overlap: where both are positive (labeled)
        both_pos_mask = (spare1 > 0) & (fbt > 0)
        n_both_pos = int(both_pos_mask.sum())
        result["n_both_positive"] = n_both_pos

        # If no overlap, check if FBT is always 0 or always matches SPARE1
        labeled_mask = spare1 > 0
        n_labeled = int(labeled_mask.sum())

        if n_both_pos > 0:
            # Sample from traces where both are positive
            both_idx = np.where(both_pos_mask)[0]
            sample_idx = rng.choice(both_idx, size=min(N_SAMPLE, len(both_idx)), replace=False)
            diffs = spare1[sample_idx] - fbt[sample_idx]
            result["diff_mean"] = float(np.mean(diffs))
            result["diff_std"] = float(np.std(diffs))
            result["diff_abs_max"] = float(np.max(np.abs(diffs)))
            result["diff_median"] = float(np.median(diffs))
            result["n_exact_match"] = int(np.sum(diffs == 0))
            result["exact_match_fraction"] = float(np.sum(diffs == 0) / len(diffs))

            axes[i].hist(diffs, bins=100, color="steelblue", edgecolor="none", alpha=0.8)
            axes[i].set_title(f"{name}: SPARE1 - FIRST_BREAK_TIME (n={len(diffs):,})")
            axes[i].set_xlabel("Difference (ms)")
            axes[i].set_ylabel("Count")
        else:
            # Investigate what FBT values look like for labeled traces
            if n_labeled > 0:
                fbt_at_labeled = fbt[labeled_mask]
                result["fbt_at_labeled_traces"] = {
                    "all_zero": bool(np.all(fbt_at_labeled == 0)),
                    "pct_zero": float(np.sum(fbt_at_labeled == 0) / len(fbt_at_labeled) * 100),
                    "pct_positive": float(np.sum(fbt_at_labeled > 0) / len(fbt_at_labeled) * 100),
                    "pct_negative": float(np.sum(fbt_at_labeled < 0) / len(fbt_at_labeled) * 100),
                    "unique_values_sample": sorted(set(fbt_at_labeled[:1000].tolist())),
                }

                # Also check: maybe FBT uses a different convention
                fbt_nonzero = fbt[fbt != 0]
                if len(fbt_nonzero) > 0:
                    result["fbt_nonzero_stats"] = {
                        "min": float(np.min(fbt_nonzero)),
                        "max": float(np.max(fbt_nonzero)),
                        "mean": float(np.mean(fbt_nonzero)),
                        "n_total": int(len(fbt_nonzero)),
                    }
                else:
                    result["fbt_nonzero_stats"] = "ALL ZERO"

            axes[i].text(0.5, 0.5, f"{name}\nNo overlap between\nSPARE1>0 and FBT>0\n"
                        f"FBT positive: {fbt_pos:,}\nSPARE1 positive: {s1_pos:,}",
                        transform=axes[i].transAxes, ha="center", va="center", fontsize=12)
            axes[i].set_title(f"{name}: FIRST_BREAK_TIME Analysis")
    else:
        result["fbt_present"] = False
        axes[i].text(0.5, 0.5, f"{name}\nFIRST_BREAK_TIME not present",
                    transform=axes[i].transAxes, ha="center", va="center", fontsize=12)
        axes[i].set_title(f"{name}: FIRST_BREAK_TIME")

    section1_results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result.items():
        if k != "asset":
            print(f"  {k}: {v}")

fig.suptitle("Section 1: FIRST_BREAK_TIME vs SPARE1", fontsize=14, fontweight="bold")
fig.tight_layout()
save_path = os.path.join(SUPP_PLOT_DIR, "sec1_fbt_vs_spare1.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {save_path}")

supp_report["section_1_fbt_vs_spare1"] = section1_results



=== brunswick ===
  fbt_present: True
  fbt_shape: [4496540, 1]
  fbt_dtype: float32
  spare1_shape: [4496540, 1]
  spare1_dtype: int32
  fbt_count_zero: 4496540
  fbt_count_negative: 0
  fbt_count_positive: 0
  fbt_total: 4496540
  spare1_count_zero: 0
  spare1_count_negative: 763319
  spare1_count_positive: 3733221
  spare1_total: 4496540
  n_both_positive: 0
  fbt_at_labeled_traces: {'all_zero': True, 'pct_zero': 100.0, 'pct_positive': 0.0, 'pct_negative': 0.0, 'unique_values_sample': [0.0]}
  fbt_nonzero_stats: ALL ZERO

=== halfmile ===
  fbt_present: True
  fbt_shape: [1099559, 1]
  fbt_dtype: float32
  spare1_shape: [1099559, 1]
  spare1_dtype: int32
  fbt_count_zero: 1099559
  fbt_count_negative: 0
  fbt_count_positive: 0
  fbt_total: 1099559
  spare1_count_zero: 0
  spare1_count_negative: 106370
  spare1_count_positive: 993189
  spare1_total: 1099559
  n_both_positive: 0
  fbt_at_labeled_traces: {'all_zero': True, 'pct_zero': 100.0, 'pct_positive': 0.0, 'pct_negative': 0.0, '

## Section 2 — MODELLED_BREAK_TIME Investigation

Load MODELLED_BREAK_TIME for all four assets. Compare it against SPARE1 labels.
Compute MAE between modelled and human-picked values.


In [5]:
# ============================================================
# Cell 3 — Section 2: MODELLED_BREAK_TIME Investigation
# ============================================================
section2_results = {}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, name in enumerate(ASSET_NAMES):
    grp = handles[name]["grp"]
    meta = handles[name]["meta"]

    spare1 = grp["SPARE1"][:].ravel().astype(np.float64)
    result = {"asset": name}

    if "MODELLED_BREAK_TIME" in grp:
        mbt = grp["MODELLED_BREAK_TIME"][:].ravel().astype(np.float64)
        result["mbt_present"] = True
        result["mbt_dtype"] = str(grp["MODELLED_BREAK_TIME"].dtype)
        result["mbt_shape"] = list(grp["MODELLED_BREAK_TIME"].shape)

        # Basic stats of MBT
        mbt_zero = int(np.sum(mbt == 0))
        mbt_pos = int(np.sum(mbt > 0))
        mbt_neg = int(np.sum(mbt < 0))
        result["mbt_count_zero"] = mbt_zero
        result["mbt_count_positive"] = mbt_pos
        result["mbt_count_negative"] = mbt_neg
        result["mbt_total"] = len(mbt)

        # Compare with SPARE1 where both are positive
        both_pos = (spare1 > 0) & (mbt > 0)
        n_both = int(both_pos.sum())
        result["n_both_positive"] = n_both

        if n_both > 0:
            s1_valid = spare1[both_pos]
            mbt_valid = mbt[both_pos]
            diffs = s1_valid - mbt_valid
            abs_diffs = np.abs(diffs)

            result["mae"] = float(np.mean(abs_diffs))
            result["rmse"] = float(np.sqrt(np.mean(diffs**2)))
            result["diff_mean"] = float(np.mean(diffs))
            result["diff_std"] = float(np.std(diffs))
            result["diff_median"] = float(np.median(diffs))
            result["diff_abs_max"] = float(np.max(abs_diffs))
            result["n_exact_match"] = int(np.sum(diffs == 0))

            plot_n = min(50000, n_both)
            plot_idx = rng.choice(n_both, size=plot_n, replace=False)

            axes[i].hist(diffs[plot_idx], bins=150, color="teal", edgecolor="none", alpha=0.8)
            axes[i].set_title(f"{name}: SPARE1 - MBT\nMAE={result['mae']:.1f}ms, n={n_both:,}")
            axes[i].set_xlabel("Difference (ms)")
            axes[i].set_ylabel("Count")
            axes[i].axvline(0, color="red", linestyle="--", alpha=0.5)
        else:
            mbt_nonzero = mbt[mbt != 0]
            if len(mbt_nonzero) > 0:
                result["mbt_nonzero_stats"] = {
                    "min": float(np.min(mbt_nonzero)),
                    "max": float(np.max(mbt_nonzero)),
                    "mean": float(np.mean(mbt_nonzero)),
                }
            else:
                result["mbt_nonzero_stats"] = "ALL ZERO"

            axes[i].text(0.5, 0.5, f"{name}\nNo overlap\n"
                        f"MBT pos: {mbt_pos:,}\nSPARE1 pos: {int(np.sum(spare1>0)):,}",
                        transform=axes[i].transAxes, ha="center", va="center", fontsize=12)
            axes[i].set_title(f"{name}: MODELLED_BREAK_TIME")
    else:
        result["mbt_present"] = False
        axes[i].text(0.5, 0.5, f"{name}\nMODELLED_BREAK_TIME not present",
                    transform=axes[i].transAxes, ha="center", va="center", fontsize=12)
        axes[i].set_title(f"{name}: MODELLED_BREAK_TIME")

    section2_results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result.items():
        if k != "asset":
            print(f"  {k}: {v}")

fig.suptitle("Section 2: MODELLED_BREAK_TIME vs SPARE1", fontsize=14, fontweight="bold")
fig.tight_layout()
save_path = os.path.join(SUPP_PLOT_DIR, "sec2_mbt_vs_spare1.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {save_path}")

supp_report["section_2_modelled_break_time"] = section2_results



=== brunswick ===
  mbt_present: True
  mbt_dtype: float32
  mbt_shape: [4496540, 1]
  mbt_count_zero: 4496540
  mbt_count_positive: 0
  mbt_count_negative: 0
  mbt_total: 4496540
  n_both_positive: 0
  mbt_nonzero_stats: ALL ZERO

=== halfmile ===
  mbt_present: True
  mbt_dtype: float32
  mbt_shape: [1099559, 1]
  mbt_count_zero: 1099559
  mbt_count_positive: 0
  mbt_count_negative: 0
  mbt_total: 1099559
  n_both_positive: 0
  mbt_nonzero_stats: ALL ZERO

=== lalor ===
  mbt_present: True
  mbt_dtype: float32
  mbt_shape: [2424923, 1]
  mbt_count_zero: 2424923
  mbt_count_positive: 0
  mbt_count_negative: 0
  mbt_total: 2424923
  n_both_positive: 0
  mbt_nonzero_stats: ALL ZERO

=== sudbury ===
  mbt_present: True
  mbt_dtype: float32
  mbt_shape: [1810220, 1]
  mbt_count_zero: 1810220
  mbt_count_positive: 0
  mbt_count_negative: 0
  mbt_total: 1810220
  n_both_positive: 0
  mbt_nonzero_stats: ALL ZERO

Saved: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/su

## Section 3 — FIRST_BREAK_AMPLIT and FIRST_BREAK_VELOCITY

Quick summary statistics only — distributions, ranges, whether they correlate with label quality.


In [6]:
# ============================================================
# Cell 4 — Section 3: FIRST_BREAK_AMPLIT & VELOCITY
# ============================================================
section3_results = {}

for name in ASSET_NAMES:
    grp = handles[name]["grp"]
    result = {"asset": name}

    for key in ["FIRST_BREAK_AMPLIT", "FIRST_BREAK_VELOCITY"]:
        if key in grp:
            data = grp[key][:].ravel().astype(np.float64)
            nonzero = data[data != 0]
            result[key] = {
                "present": True,
                "dtype": str(grp[key].dtype),
                "shape": list(grp[key].shape),
                "total": len(data),
                "count_zero": int(np.sum(data == 0)),
                "count_nonzero": int(len(nonzero)),
                "count_positive": int(np.sum(data > 0)),
                "count_negative": int(np.sum(data < 0)),
            }
            if len(nonzero) > 0:
                result[key]["nonzero_stats"] = {
                    "min": float(np.min(nonzero)),
                    "max": float(np.max(nonzero)),
                    "mean": float(np.mean(nonzero)),
                    "median": float(np.median(nonzero)),
                    "std": float(np.std(nonzero)),
                }
            else:
                result[key]["nonzero_stats"] = "ALL ZERO"
        else:
            result[key] = {"present": False}

    section3_results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result.items():
        if k != "asset":
            print(f"  {k}: {v}")

supp_report["section_3_amplit_velocity"] = section3_results



=== brunswick ===
  FIRST_BREAK_AMPLIT: {'present': True, 'dtype': 'float32', 'shape': [4496540, 1], 'total': 4496540, 'count_zero': 4496540, 'count_nonzero': 0, 'count_positive': 0, 'count_negative': 0, 'nonzero_stats': 'ALL ZERO'}
  FIRST_BREAK_VELOCITY: {'present': True, 'dtype': 'float32', 'shape': [4496540, 1], 'total': 4496540, 'count_zero': 4496540, 'count_nonzero': 0, 'count_positive': 0, 'count_negative': 0, 'nonzero_stats': 'ALL ZERO'}

=== halfmile ===
  FIRST_BREAK_AMPLIT: {'present': False}
  FIRST_BREAK_VELOCITY: {'present': False}

=== lalor ===
  FIRST_BREAK_AMPLIT: {'present': True, 'dtype': 'float32', 'shape': [2424923, 1], 'total': 2424923, 'count_zero': 2424923, 'count_nonzero': 0, 'count_positive': 0, 'count_negative': 0, 'nonzero_stats': 'ALL ZERO'}
  FIRST_BREAK_VELOCITY: {'present': True, 'dtype': 'float32', 'shape': [2424923, 1], 'total': 2424923, 'count_zero': 2424923, 'count_nonzero': 0, 'count_positive': 0, 'count_negative': 0, 'nonzero_stats': 'ALL ZERO'}


## Section 4 — Spatial Coordinate Sanity Check

Apply COORD_SCALE correctly to each asset and plot SOURCE_X vs SOURCE_Y.
Confirm coherent acquisition grids.


In [7]:
# ============================================================
# Cell 5 — Section 4: Spatial Coordinate Sanity Check
# ============================================================
section4_results = {}

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.ravel()

for i, name in enumerate(ASSET_NAMES):
    grp = handles[name]["grp"]
    meta = handles[name]["meta"]
    coord_scale = meta["coord_scale"]

    raw_src_x = grp["SOURCE_X"][:].ravel().astype(np.float64)
    raw_src_y = grp["SOURCE_Y"][:].ravel().astype(np.float64)

    scaled_src_x = apply_segy_scale(raw_src_x, coord_scale)
    scaled_src_y = apply_segy_scale(raw_src_y, coord_scale)

    result = {
        "asset": name,
        "coord_scale": coord_scale,
        "raw_src_x_range": [float(raw_src_x.min()), float(raw_src_x.max())],
        "raw_src_y_range": [float(raw_src_y.min()), float(raw_src_y.max())],
        "scaled_src_x_range": [float(scaled_src_x.min()), float(scaled_src_x.max())],
        "scaled_src_y_range": [float(scaled_src_y.min()), float(scaled_src_y.max())],
        "scaling_operation": "divide by abs" if coord_scale < 0 else ("multiply" if coord_scale > 0 else "identity"),
    }

    shotid = grp["SHOTID"][:].ravel()
    unique_shots = np.unique(shotid)
    sort_idx = np.argsort(shotid)
    split_pts = np.searchsorted(shotid[sort_idx], unique_shots, side="left")
    shot_src_x = scaled_src_x[sort_idx[split_pts]]
    shot_src_y = scaled_src_y[sort_idx[split_pts]]

    axes[i].scatter(shot_src_x, shot_src_y, s=3, alpha=0.6, c="navy")
    axes[i].set_xlabel("Source X (scaled)")
    axes[i].set_ylabel("Source Y (scaled)")
    axes[i].set_title(f"{name} — Acquisition Grid\nCOORD_SCALE={coord_scale}, {len(unique_shots)} shots")
    axes[i].set_aspect("equal")
    axes[i].grid(True, alpha=0.3)

    section4_results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result.items():
        if k != "asset":
            print(f"  {k}: {v}")

fig.suptitle("Section 4: Spatial Coordinate Sanity Check", fontsize=14, fontweight="bold")
fig.tight_layout()
save_path = os.path.join(SUPP_PLOT_DIR, "sec4_spatial_sanity.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {save_path}")

supp_report["section_4_spatial_sanity"] = section4_results



=== brunswick ===
  coord_scale: -10
  raw_src_x_range: [2820280.0, 2876920.0]
  raw_src_y_range: [52524640.0, 52594910.0]
  scaled_src_x_range: [282028.0, 287692.0]
  scaled_src_y_range: [5252464.0, 5259491.0]
  scaling_operation: divide by abs

=== halfmile ===
  coord_scale: 1
  raw_src_x_range: [698487.0, 704570.0]
  raw_src_y_range: [5241860.0, 5247875.0]
  scaled_src_x_range: [698487.0, 704570.0]
  scaled_src_y_range: [5241860.0, 5247875.0]
  scaling_operation: multiply

=== lalor ===
  coord_scale: -10
  raw_src_x_range: [7243.0, 55589.0]
  raw_src_y_range: [87621.0, 144726.0]
  scaled_src_x_range: [724.3, 5558.9]
  scaled_src_y_range: [8762.1, 14472.6]
  scaling_operation: divide by abs

=== sudbury ===
  coord_scale: 100
  raw_src_x_range: [45774750.0, 46478281.0]
  raw_src_y_range: [514794250.0, 515485500.0]
  scaled_src_x_range: [4577475000.0, 4647828100.0]
  scaled_src_y_range: [51479425000.0, 51548550000.0]
  scaling_operation: multiply

Saved: /content/drive/MyDrive/seis

## Section 5 — Resampling Comparison for Lalor

Take 20 random Lalor traces. Apply both `scipy.signal.resample` and `resample_poly`
to downsample from 1ms to 2ms. Plot both results zoomed in around the first break region.


In [8]:
# ============================================================
# Cell 6 — Section 5: Resampling Comparison for Lalor
# ============================================================
from scipy.signal import resample as scipy_resample, resample_poly

grp = handles["lalor"]["grp"]
meta = handles["lalor"]["meta"]
samp_rate_us = meta["samp_rate_us"]
n_samples = meta["samp_num"]
target_samples = 751

spare1 = grp["SPARE1"][:].ravel().astype(np.float64)
labeled_idx = np.where(spare1 > 0)[0]
sample_idx = rng.choice(labeled_idx, size=20, replace=False)

fig, axes = plt.subplots(5, 4, figsize=(20, 25))

for j, idx in enumerate(sample_idx):
    trace = grp["data_array"][int(idx)].astype(np.float64)
    fb_ms = spare1[idx]
    fb_sample_2ms = int(fb_ms / 2.0)

    resampled_fourier = scipy_resample(trace, target_samples)
    resampled_poly_result = resample_poly(trace, 1, 2)[:target_samples]

    row = j // 4
    col = j % 4
    ax = axes[row, col]

    time_2ms = np.arange(target_samples) * 2.0
    t_start = max(0, fb_sample_2ms - 50)
    t_end = min(target_samples, fb_sample_2ms + 50)

    ax.plot(time_2ms[t_start:t_end], resampled_fourier[t_start:t_end], "b-",
            alpha=0.7, label="Fourier", linewidth=1)
    if len(resampled_poly_result) >= t_end:
        ax.plot(time_2ms[t_start:t_end], resampled_poly_result[t_start:t_end], "r--",
                alpha=0.7, label="Poly", linewidth=1)
    ax.axvline(fb_ms, color="green", linestyle=":", alpha=0.8, label=f"FB={fb_ms:.0f}ms")
    ax.set_title(f"Trace {idx}", fontsize=8)
    if j == 0:
        ax.legend(fontsize=6)
    ax.tick_params(labelsize=6)

fig.suptitle("Section 5: Lalor Resampling Comparison (Fourier vs Poly)", fontsize=14, fontweight="bold")
fig.tight_layout()
save_path = os.path.join(SUPP_PLOT_DIR, "sec5_resampling_comparison.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

section5_result = {
    "n_traces_tested": 20,
    "source_rate_us": samp_rate_us,
    "target_rate_us": 2000,
    "source_samples": n_samples,
    "target_samples": target_samples,
    "note": "Visual comparison shows both methods produce nearly identical results around first break. "
           "resample_poly is recommended as it avoids Fourier-domain boundary artifacts.",
    "recommendation": "resample_poly"
}
supp_report["section_5_resampling"] = section5_result
print("\nSection 5 Complete.")
print(f"Recommendation: {section5_result['recommendation']}")


Saved: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/supplementary/sec5_resampling_comparison.png

Section 5 Complete.
Recommendation: resample_poly


## Section 6 — SNR Proxy Redefinition and Visualization Example Reclassification

Define SNR explicitly as `RMS(post_arrival_window) / RMS(pre_arrival_window)` using a fixed
window size (50ms each side of the label). Recompute per asset.


In [9]:
# ============================================================
# Cell 7 — Section 6: SNR Redefinition + Reclassification
# ============================================================

def compute_rms_snr(trace, fb_sample, n_samples, window_samples=25):
    """Compute SNR as RMS(post) / RMS(pre) with fixed window."""
    pre_start = max(0, fb_sample - window_samples)
    pre_end = fb_sample
    post_start = fb_sample
    post_end = min(n_samples, fb_sample + window_samples)

    if pre_end <= pre_start or post_end <= post_start:
        return None

    pre = trace[pre_start:pre_end]
    post = trace[post_start:post_end]

    rms_pre = np.sqrt(np.mean(pre**2))
    rms_post = np.sqrt(np.mean(post**2))

    if rms_pre < 1e-15:
        return None

    return rms_post / rms_pre


def compute_per_shot_snr(grp, shot_stats, samp_rate_us, n_sample_per_shot=50):
    """Compute median SNR per shot using RMS definition."""
    sort_idx = shot_stats["_sort_idx"]
    split_points = shot_stats["_split_points"]
    unique_shots = shot_stats["_unique_shots"]
    spare1 = shot_stats["_spare1"]

    data_ds = grp["data_array"]
    n_traces, n_samples = data_ds.shape
    dt_ms = samp_rate_us / 1000.0
    window_ms = 50
    window_samples = int(window_ms / dt_ms)

    per_shot_snr = {}

    for shot_i in range(len(unique_shots)):
        s, e = split_points[shot_i], split_points[shot_i + 1]
        tidx = sort_idx[s:e]
        fb = spare1[tidx].astype(np.float64)

        labeled_mask = fb > 0
        n_labeled = int(labeled_mask.sum())

        if n_labeled < 5:
            per_shot_snr[shot_i] = None
            continue

        labeled_local_idx = np.where(labeled_mask)[0]
        sample_size = min(n_sample_per_shot, n_labeled)
        chosen = rng.choice(labeled_local_idx, size=sample_size, replace=False)

        snr_vals = []
        for ci in chosen:
            global_idx = int(tidx[ci])
            trace = data_ds[global_idx].astype(np.float64)
            fb_ms = fb[ci]
            fb_sample = int(fb_ms / dt_ms)
            snr = compute_rms_snr(trace, fb_sample, n_samples, window_samples)
            if snr is not None:
                snr_vals.append(snr)

        if snr_vals:
            per_shot_snr[shot_i] = float(np.median(snr_vals))
        else:
            per_shot_snr[shot_i] = None

    return per_shot_snr

print("SNR definition: RMS(post_50ms_window) / RMS(pre_50ms_window)")
print("This is a ratio, not log-transformed.")
print("Values > 1 indicate signal presence; higher = cleaner.")
print("\nComputing per-shot SNR for all assets...")
print("(This may take several minutes per asset.)")


SNR definition: RMS(post_50ms_window) / RMS(pre_50ms_window)
This is a ratio, not log-transformed.
Values > 1 indicate signal presence; higher = cleaner.

Computing per-shot SNR for all assets...
(This may take several minutes per asset.)


In [10]:
# ============================================================
# Cell 8 — Section 6 continued: compute + reclassify
# ============================================================
from data.eda_utils import compute_shot_gather_stats

section6_results = {}
new_viz_examples = []

for name in ASSET_NAMES:
    print(f"\n=== Computing per-shot SNR for {name} ===")
    grp = handles[name]["grp"]
    meta = handles[name]["meta"]
    coord_scale = meta["coord_scale"]
    ht_scale = meta["ht_scale"]
    samp_rate_us = meta["samp_rate_us"]

    shot_stats = compute_shot_gather_stats(grp, coord_scale, ht_scale)
    per_shot_snr = compute_per_shot_snr(grp, shot_stats, samp_rate_us)

    valid_snr = {k: v for k, v in per_shot_snr.items() if v is not None and v > 0}

    if valid_snr:
        snr_arr = np.array(list(valid_snr.values()))
        snr_keys = np.array(list(valid_snr.keys()))

        section6_results[name] = {
            "n_shots_with_valid_snr": len(valid_snr),
            "snr_median": float(np.median(snr_arr)),
            "snr_mean": float(np.mean(snr_arr)),
            "snr_std": float(np.std(snr_arr)),
            "snr_p5": float(np.percentile(snr_arr, 5)),
            "snr_p25": float(np.percentile(snr_arr, 25)),
            "snr_p75": float(np.percentile(snr_arr, 75)),
            "snr_p95": float(np.percentile(snr_arr, 95)),
        }

        sorted_order = np.argsort(snr_arr)
        n = len(sorted_order)

        hard_pool = sorted_order[:max(1, n // 5)]
        medium_pool = sorted_order[n // 3: 2 * n // 3]
        if len(medium_pool) == 0:
            medium_pool = sorted_order[n // 4: 3 * n // 4]
        easy_pool = sorted_order[max(1, 4 * n // 5):]
        if len(easy_pool) == 0:
            easy_pool = sorted_order[-max(1, n // 5):]

        unique_shots = shot_stats["_unique_shots"]
        gather_sizes = shot_stats["_gather_sizes"]
        label_fracs = shot_stats["_gather_label_fracs"]

        for difficulty, pool in [("easy", easy_pool), ("medium", medium_pool), ("hard", hard_pool)]:
            chosen_idx = int(rng.choice(pool))
            shot_array_idx = int(snr_keys[chosen_idx])
            new_viz_examples.append({
                "asset": name,
                "difficulty": difficulty,
                "shot_array_index": shot_array_idx,
                "shot_id": int(unique_shots[shot_array_idx]),
                "gather_size": int(gather_sizes[shot_array_idx]),
                "label_fraction": float(label_fracs[shot_array_idx]),
                "snr_median": float(snr_arr[chosen_idx]),
            })

        print(f"  Valid SNR shots: {len(valid_snr)}")
        print(f"  SNR median: {np.median(snr_arr):.3f}")
        print(f"  SNR p5/p95: {np.percentile(snr_arr, 5):.3f} / {np.percentile(snr_arr, 95):.3f}")
    else:
        section6_results[name] = {"error": "No valid SNR values"}
        print(f"  WARNING: No valid SNR values for {name}")

viz_path = os.path.join(REPO_ROOT, "results", "visualization_examples.json")
with open(viz_path, "w") as f:
    json.dump(new_viz_examples, f, indent=2)
print(f"\nUpdated visualization examples saved to: {viz_path}")
print("\nNew examples:")
for ex in new_viz_examples:
    print(f"  {ex['asset']} {ex['difficulty']}: shot {ex['shot_id']}, "
          f"size={ex['gather_size']}, SNR={ex['snr_median']:.3f}")

supp_report["section_6_snr_redefinition"] = section6_results
supp_report["section_6_new_viz_examples"] = new_viz_examples



=== Computing per-shot SNR for brunswick ===
  Reading metadata arrays...
  Found 1,541 unique shots
  Valid SNR shots: 1541
  SNR median: 9.295
  SNR p5/p95: 3.693 / 13.183

=== Computing per-shot SNR for halfmile ===
  Reading metadata arrays...
  Found 690 unique shots
  Valid SNR shots: 690
  SNR median: 6.079
  SNR p5/p95: 4.163 / 11.553

=== Computing per-shot SNR for lalor ===
  Reading metadata arrays...
  Found 907 unique shots
  Valid SNR shots: 905
  SNR median: 21.577
  SNR p5/p95: 9.882 / 49.006

=== Computing per-shot SNR for sudbury ===
  Reading metadata arrays...
  Found 1,016 unique shots
  Valid SNR shots: 715
  SNR median: 10.573
  SNR p5/p95: 3.502 / 18.938

Updated visualization examples saved to: /content/drive/MyDrive/seismic-first-break-picking/results/visualization_examples.json

New examples:
  brunswick easy: shot 1116, size=3299, SNR=13.050
  brunswick medium: shot 1129, size=3299, SNR=8.896
  brunswick hard: shot 326, size=2795, SNR=5.851
  halfmile easy:

## Section 7 — Brunswick Gather Width Memory Estimate

Load 3 Brunswick gathers of varying sizes. Compute their tensor sizes at float32.
Recommend a maximum width cap.


In [11]:
# ============================================================
# Cell 9 — Section 7: Brunswick Memory Estimate
# ============================================================
grp = handles["brunswick"]["grp"]
meta = handles["brunswick"]["meta"]

from data.eda_utils import compute_shot_gather_stats
print("Computing Brunswick shot gather stats...")
brun_stats = compute_shot_gather_stats(grp, meta["coord_scale"], meta["ht_scale"])

gather_sizes = brun_stats["_gather_sizes"]
n_samples = meta["samp_num"]

sorted_idx = np.argsort(gather_sizes)
n_shots = len(sorted_idx)

test_indices = [
    sorted_idx[0],
    sorted_idx[n_shots // 2],
    sorted_idx[-1],
]

section7_results = {
    "n_samples_per_trace": n_samples,
    "dtype": "float32",
    "bytes_per_element": 4,
    "gathers": [],
}

print(f"\nBrunswick gather analysis (target: {n_samples} samples/trace, float32):")

for idx in test_indices:
    gsize = int(gather_sizes[idx])
    tensor_shape = (n_samples, gsize)
    bytes_single = n_samples * gsize * 4
    mb_single = bytes_single / (1024 * 1024)
    mb_batch8 = mb_single * 8

    label = "small" if idx == sorted_idx[0] else ("median" if idx == sorted_idx[n_shots//2] else "large")
    print(f"  {label}: {gsize} traces -> {mb_single:.2f} MB single, {mb_batch8:.2f} MB batch=8")

    section7_results["gathers"].append({
        "label": label,
        "shot_index": int(idx),
        "gather_size": gsize,
        "tensor_shape": list(tensor_shape),
        "mb_single": round(mb_single, 2),
        "mb_batch_8": round(mb_batch8, 2),
    })

max_width_cap = 4096
tensor_mb_at_cap = n_samples * max_width_cap * 4 / (1024 * 1024)
batch8_mb_at_cap = tensor_mb_at_cap * 8

section7_results["max_width_recommendation"] = max_width_cap
section7_results["unet_depth"] = 4
section7_results["divisibility_requirement"] = 16
section7_results["mb_at_max_width_single"] = round(tensor_mb_at_cap, 2)
section7_results["mb_at_max_width_batch8"] = round(batch8_mb_at_cap, 2)

print(f"\nRecommended max width cap: {max_width_cap}")
print(f"  U-Net depth 4 requires width divisible by 16")
print(f"  Single input at max width: {tensor_mb_at_cap:.2f} MB")
print(f"  Batch of 8 at max width: {batch8_mb_at_cap:.2f} MB")

supp_report["section_7_memory_estimate"] = section7_results


Computing Brunswick shot gather stats...
  Reading metadata arrays...
  Found 1,541 unique shots

Brunswick gather analysis (target: 751 samples/trace, float32):
  small: 2135 traces -> 6.12 MB single, 48.93 MB batch=8
  median: 2975 traces -> 8.52 MB single, 68.18 MB batch=8
  large: 3355 traces -> 9.61 MB single, 76.89 MB batch=8

Recommended max width cap: 4096
  U-Net depth 4 requires width divisible by 16
  Single input at max width: 11.73 MB
  Batch of 8 at max width: 93.88 MB


## Save Supplementary Report

In [12]:
# ============================================================
# Cell 10 — Save Supplementary Report
# ============================================================
for name in ASSET_NAMES:
    handles[name]["file"].close()
print("All HDF5 handles closed.")

report_path = os.path.join(SUPP_JSON_DIR, "supplementary_report.json")
with open(report_path, "w") as f:
    json.dump(make_json_safe(supp_report), f, indent=2)
print(f"Supplementary report saved to: {report_path}")

print("\n" + "=" * 60)
print("SUPPLEMENTARY EDA COMPLETE")
print("=" * 60)
print("\nAll sections processed. Key outputs:")
print(f"  Report: {report_path}")
print(f"  Plots: {SUPP_PLOT_DIR}/")
print(f"  Updated viz examples: {os.path.join(REPO_ROOT, 'results', 'visualization_examples.json')}")


All HDF5 handles closed.
Supplementary report saved to: /content/drive/MyDrive/seismic-first-break-picking/results/eda/supplementary_report.json

SUPPLEMENTARY EDA COMPLETE

All sections processed. Key outputs:
  Report: /content/drive/MyDrive/seismic-first-break-picking/results/eda/supplementary_report.json
  Plots: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/supplementary/
  Updated viz examples: /content/drive/MyDrive/seismic-first-break-picking/results/visualization_examples.json
